In [ ]:
import pandas as pd

df = pd.read_csv("Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")
print(df.shape)
df.head()

(26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [3]:
df.info()
print(df.isnull().sum())
print("Duplicated rows:", df.duplicated().sum())
print("Duplicated instructions:", df["instruction"].duplicated().sum())


<class 'pandas.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   flags        26872 non-null  str  
 1   instruction  26872 non-null  str  
 2   category     26872 non-null  str  
 3   intent       26872 non-null  str  
 4   response     26872 non-null  str  
dtypes: str(5)
memory usage: 1.0 MB
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64
Duplicated rows: 0
Duplicated instructions: 2237


In [4]:
dup_mask = df["instruction"].duplicated(keep=False)
dups = df[dup_mask]

conflicts = dups.groupby("instruction")["intent"].nunique()
print("Duplicated instruction groups:", conflicts.shape[0])
print("Groups with conflicting intents:", (conflicts > 1).sum())

Duplicated instruction groups: 989
Groups with conflicting intents: 0


In [5]:
import re

def clean_text(text):
    # 1) توحيد الـ placeholders: {{Order Number}} -> ph_order_number
    text = re.sub(
        r"\{\{(.*?)\}\}",
        lambda m: "ph_" + re.sub(r"\s+", "_", m.group(1).strip().lower()),
        text,
    )
    # 2) lowercase
    text = text.lower()
    # 3) تنضيف المسافات الزيادة
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["instruction_clean"] = df["instruction"].apply(clean_text)

df[["instruction", "instruction_clean"]].head(8)

,instruction,instruction_clean
0,question about cancelling order {{Order Number}},question about cancelling order ph_order_number
1,i have a question about cancelling oorder {{Or...,i have a question about cancelling oorder ph_o...
2,i need help cancelling puchase {{Order Number}},i need help cancelling puchase ph_order_number
3,I need to cancel purchase {{Order Number}},i need to cancel purchase ph_order_number
4,"I cannot afford this order, cancel purchase {{...","i cannot afford this order, cancel purchase ph..."
5,can you help me cancel order {{Order Number}}?,can you help me cancel order ph_order_number?
6,"I can no longer afford order {{Order Number}},...","i can no longer afford order ph_order_number, ..."
7,I am trying to cancel purchase {{Order Number}},i am trying to cancel purchase ph_order_number


In [6]:
before = len(df)
df_dedup = df.drop_duplicates(subset="instruction_clean", keep="first").reset_index(drop=True)

print("Before:", before)
print("After :", len(df_dedup))
print("Removed:", before - len(df_dedup))

Before: 26872
After : 24274
Removed: 2598


In [7]:
df_dedup["intent"].value_counts().describe()

count     27.000000
mean     899.037037
std      133.974303
min      436.000000
25%      865.500000
50%      961.000000
75%      991.000000
max      997.000000
Name: count, dtype: float64

In [8]:
before = df["intent"].value_counts()
after = df_dedup["intent"].value_counts()

comp = pd.DataFrame({"before": before, "after": after})
comp["removed"] = comp["before"] - comp["after"]
comp.sort_values("removed", ascending=False).head(8)

,before,after,removed
intent,,,
cancel_order,998,436,562
delivery_options,995,655,340
track_refund,998,692,306
edit_account,1000,766,234
track_order,995,783,212
switch_account,1000,836,164
change_order,997,856,141
create_account,997,875,122


In [9]:
from sklearn.model_selection import train_test_split
import os

train, temp = train_test_split(
    df_dedup, test_size=0.2, stratify=df_dedup["intent"], random_state=42
)
val, test = train_test_split(
    temp, test_size=0.5, stratify=temp["intent"], random_state=42
)

print("Train:", len(train), "| Val:", len(val), "| Test:", len(test))

os.makedirs("processed_data", exist_ok=True)
train.to_csv("processed_data/train.csv", index=False)
val.to_csv("processed_data/val.csv", index=False)
test.to_csv("processed_data/test.csv", index=False)

Train: 19419 | Val: 2427 | Test: 2428


In [10]:
print(train["intent"].value_counts(normalize=True).head(3))
print(test["intent"].value_counts(normalize=True).head(3))

intent
complaint                0.041042
place_order              0.040888
registration_problems    0.040888
Name: proportion, dtype: float64
intent
review                   0.041186
check_payment_methods    0.041186
payment_issue            0.041186
Name: proportion, dtype: float64


In [11]:
train_set = set(train["instruction_clean"])
print("Train/Test overlap:", len(train_set & set(test["instruction_clean"])))
print("Train/Val overlap :", len(train_set & set(val["instruction_clean"])))

Train/Test overlap: 0
Train/Val overlap : 0


In [12]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

train = pd.read_csv("processed_data/train.csv")
val = pd.read_csv("processed_data/val.csv")

# كلمات نفي وأسئلة ممكن تفرق في المعنى، فنحافظ عليها
keep = {"not", "no", "nor", "cannot", "can", "how", "what", "where", "when", "why", "who"}
custom_stop = list(ENGLISH_STOP_WORDS - keep)

def run(stop_words, name):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, stop_words=stop_words)
    X_train = vec.fit_transform(train["instruction_clean"])
    X_val = vec.transform(val["instruction_clean"])

    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X_train, train["intent"])
    pred = clf.predict(X_val)

    return {
        "setting": name,
        "n_features": X_train.shape[1],
        "accuracy": round(accuracy_score(val["intent"], pred), 4),
        "macro_f1": round(f1_score(val["intent"], pred, average="macro"), 4),
    }

results = pd.DataFrame([
    run(None, "No stop-word removal"),
    run("english", "sklearn english stop words"),
    run(custom_stop, "Custom (keep negation + question words)"),
])
results

,setting,n_features,accuracy,macro_f1
0,No stop-word removal,4974,0.9926,0.9912
1,sklearn english stop words,4897,0.9753,0.9737
2,Custom (keep negation + question words),5310,0.9757,0.9743


### Stop words experiment (evaluated on val.csv)

| Setting | Accuracy | Macro-F1 |
|---|---|---|
| No removal | 0.9926 | 0.9912 |
| sklearn english | 0.9753 | 0.9737 |
| Custom (keep negation + question words) | 0.9757 | 0.9743 |

**Decision:** do not remove stop words.
**Reason:** sklearn's list contains intent-bearing words (bill, call, see, show, get),
and the sentences are short, so removal loses signal.

In [13]:
import re
import nltk
import pandas as pd
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

nltk.download("wordnet")
nltk.download("omw-1.4")

train = pd.read_csv("processed_data/train.csv")
val = pd.read_csv("processed_data/val.csv")

lemm = WordNetLemmatizer()
stem = PorterStemmer()

def tokens(s):
    return re.findall(r"\b\w+\b", s)

def lemmatize(s):
    # منلمسش الـ placeholders زي ph_order_number
    return " ".join(w if w.startswith("ph_") else lemm.lemmatize(w) for w in tokens(s))

def stem_text(s):
    return " ".join(w if w.startswith("ph_") else stem.stem(w) for w in tokens(s))

def run(tr_text, va_text, name, **vec_kwargs):
    vec = TfidfVectorizer(sublinear_tf=True, **vec_kwargs)
    X_train = vec.fit_transform(tr_text)
    X_val = vec.transform(va_text)

    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X_train, train["intent"])
    pred = clf.predict(X_val)

    return {
        "setting": name,
        "n_features": X_train.shape[1],
        "accuracy": round(accuracy_score(val["intent"], pred), 4),
        "macro_f1": round(f1_score(val["intent"], pred, average="macro"), 4),
    }

raw_tr, raw_va = train["instruction_clean"], val["instruction_clean"]

results = pd.DataFrame([
    run(raw_tr, raw_va, "Word 1-2gram (baseline)", ngram_range=(1, 2), min_df=2),
    run(raw_tr.apply(lemmatize), raw_va.apply(lemmatize), "Lemmatized word 1-2gram", ngram_range=(1, 2), min_df=2),
    run(raw_tr.apply(stem_text), raw_va.apply(stem_text), "Stemmed word 1-2gram", ngram_range=(1, 2), min_df=2),
    run(raw_tr, raw_va, "Char 2-5gram (char_wb)", analyzer="char_wb", ngram_range=(2, 5), min_df=2),
])
results

[nltk_data] Downloading package wordnet to C:\Users\My
[nltk_data]     Lab\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to C:\Users\My
[nltk_data]     Lab\AppData\Roaming\nltk_data...


,setting,n_features,accuracy,macro_f1
0,Word 1-2gram (baseline),4974,0.9926,0.9912
1,Lemmatized word 1-2gram,4734,0.9922,0.9909
2,Stemmed word 1-2gram,4317,0.9918,0.9907
3,Char 2-5gram (char_wb),10106,0.9975,0.9971


In [14]:
from scipy.sparse import hstack

tr, va = train["instruction_clean"], val["instruction_clean"]

def mk_word():
    return TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2)

def mk_char():
    return TfidfVectorizer(sublinear_tf=True, analyzer="char_wb", ngram_range=(2, 5), min_df=2)

def fit_eval(vecs):
    X_train = hstack([v.fit_transform(tr) for v in vecs]).tocsr()
    X_val = hstack([v.transform(va) for v in vecs]).tocsr()
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X_train, train["intent"])
    return clf.predict(X_val), X_train.shape[1]

configs = {
    "Word": [mk_word()],
    "Char": [mk_char()],
    "Word+Char": [mk_word(), mk_char()],
}

has_typo = val["flags"].str.contains("Z")
rows = []
for name, vecs in configs.items():
    pred, n_feat = fit_eval(vecs)
    correct = pred == val["intent"]
    rows.append({
        "setting": name,
        "n_features": n_feat,
        "macro_f1": round(f1_score(val["intent"], pred, average="macro"), 4),
        "acc_all": round(correct.mean(), 4),
        "acc_typo_Z": round(correct[has_typo].mean(), 4),
        "acc_no_typo": round(correct[~has_typo].mean(), 4),
        "errors": int((~correct).sum()),
    })

pd.DataFrame(rows)

,setting,n_features,macro_f1,acc_all,acc_typo_Z,acc_no_typo,errors
0,Word,4974,0.9912,0.9926,0.9654,0.9990,18
1,Char,10106,0.9971,0.9975,0.9935,0.9985,6
2,Word+Char,15080,0.9975,0.9979,0.9935,0.9990,5


In [15]:
import os
import joblib
import pandas as pd
from scipy.sparse import save_npz
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer

train = pd.read_csv("processed_data/train.csv")
val = pd.read_csv("processed_data/val.csv")
test = pd.read_csv("processed_data/test.csv")

tfidf = FeatureUnion([
    ("word", TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2)),
    ("char", TfidfVectorizer(sublinear_tf=True, analyzer="char_wb", ngram_range=(2, 5), min_df=2)),
])

# الـ fit على الـ Train بس ⭐
X_train = tfidf.fit_transform(train["instruction_clean"])
X_val = tfidf.transform(val["instruction_clean"])
X_test = tfidf.transform(test["instruction_clean"])

print(X_train.shape, X_val.shape, X_test.shape)

os.makedirs("artifacts", exist_ok=True)
joblib.dump(tfidf, "artifacts/tfidf_word_char.joblib")
save_npz("artifacts/X_train.npz", X_train)
save_npz("artifacts/X_val.npz", X_val)
save_npz("artifacts/X_test.npz", X_test)

(19419, 15080) (2427, 15080) (2428, 15080)


In [16]:
import joblib

tfidf = joblib.load("artifacts/tfidf_word_char.joblib")
print(tfidf.transform(["i need help cancelling puchase ph_order_number"]).shape)

(1, 15080)


In [17]:
import random
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

train = pd.read_csv("processed_data/train.csv")
val = pd.read_csv("processed_data/val.csv")

def add_typos(text, p, rng):
    out = []
    for w in text.split():
        # منلمسش الـ placeholders ولا الكلمات القصيرة
        if w.startswith("ph_") or len(w) < 4 or rng.random() > p:
            out.append(w)
            continue
        i = rng.randrange(1, len(w) - 1)
        op = rng.choice(["swap", "drop", "dup", "sub"])
        if op == "swap":
            w = w[:i] + w[i + 1] + w[i] + w[i + 2:]
        elif op == "drop":
            w = w[:i] + w[i + 1:]
        elif op == "dup":
            w = w[:i] + w[i] + w[i:]
        else:
            w = w[:i] + rng.choice("abcdefghijklmnopqrstuvwxyz") + w[i + 1:]
        out.append(w)
    return " ".join(out)

levels = [0.0, 0.1, 0.2, 0.3, 0.5]
noisy_val = {}
for p in levels:
    rng = random.Random(42)
    noisy_val[p] = val["instruction_clean"].apply(lambda s: add_typos(s, p, rng))

# عينة تشوفي بيها شكل الـ typos
print(noisy_val[0.3].head(5).to_string())

def word_vec():
    return TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2)

def char_vec():
    return TfidfVectorizer(sublinear_tf=True, analyzer="char_wb", ngram_range=(2, 5), min_df=2)

setups = {
    "Word": [word_vec],
    "Char": [char_vec],
    "Word+Char": [word_vec, char_vec],
}

rows = []
for name, makers in setups.items():
    vecs = [m() for m in makers]
    X_train = hstack([v.fit_transform(train["instruction_clean"]) for v in vecs]).tocsr()
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X_train, train["intent"])

    for p in levels:
        X_val = hstack([v.transform(noisy_val[p]) for v in vecs]).tocsr()
        pred = clf.predict(X_val)
        rows.append({
            "features": name,
            "typo_rate": p,
            "macro_f1": round(f1_score(val["intent"], pred, average="macro"), 4),
        })

robust = pd.DataFrame(rows).pivot(index="typo_rate", columns="features", values="macro_f1")
robust

0      i want to locte my invoicse from ph_person_name
1            i have got to unsubscirbe to ur newsetter
2                              help to get bhll #00108
3    i wnt help reporting an isuse wiith online pay...
4    i do not know what i need to do to check the b...


features,Char,Word,Word+Char
typo_rate,,,
0.0,0.9971,0.9912,0.9975
0.1,0.9963,0.9757,0.9958
0.2,0.9947,0.9666,0.9940
0.3,0.9899,0.9333,0.9857
0.5,0.9892,0.8719,0.9820


In [18]:
import numpy as np

seeds = [1, 2, 3, 4, 5]
rows = []

for name in ["Char", "Word+Char"]:
    makers = setups[name]
    vecs = [m() for m in makers]
    X_train = hstack([v.fit_transform(train["instruction_clean"]) for v in vecs]).tocsr()
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X_train, train["intent"])

    for p in [0.1, 0.3, 0.5]:
        scores = []
        for s in seeds:
            rng = random.Random(s)
            noisy = val["instruction_clean"].apply(lambda t: add_typos(t, p, rng))
            X_val = hstack([v.transform(noisy) for v in vecs]).tocsr()
            scores.append(f1_score(val["intent"], clf.predict(X_val), average="macro"))
        rows.append({"features": name, "typo_rate": p,
                     "mean_f1": round(np.mean(scores), 4), "std": round(np.std(scores), 4)})

pd.DataFrame(rows).pivot(index="typo_rate", columns="features", values=["mean_f1", "std"])

mean_f1               std          
features     Char Word+Char    Char Word+Char
typo_rate                                    
0.1        0.9962    0.9956  0.0006    0.0016
0.3        0.9914    0.9885  0.0013    0.0027
0.5        0.9872    0.9802  0.0017    0.0009

In [19]:
import joblib

tfidf_char = TfidfVectorizer(sublinear_tf=True, analyzer="char_wb", ngram_range=(2, 5), min_df=2)

# الـ fit على الـ Train بس ⭐
X_train = tfidf_char.fit_transform(train["instruction_clean"])
print(X_train.shape)

joblib.dump(tfidf_char, "artifacts/tfidf_char.joblib")


(19419, 10106)


['artifacts/tfidf_char.joblib']

In [20]:
from preprocessing import clean_text

df = pd.read_csv("Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")
assert (df["instruction"].apply(clean_text) == df.loc[df.index, "instruction"].apply(clean_text)).all()

# المقارنة الحقيقية مع الداتا اللي اتدرب عليها الموديل
train = pd.read_csv("processed_data/train.csv")
assert (train["instruction"].apply(clean_text) == train["instruction_clean"]).all()
print("OK: same as the cleaning used for training")

OK: same as the cleaning used for training
